IMPORT MODULES

In [57]:
import pandas as pd
from collections import defaultdict
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from math import log2

LOADING TEXT

In [58]:
df = pd.read_csv("WITH.csv",encoding="latin-1",header=None)
df.columns = ["Sentence"]   #renaming the column

LEMMATIZATION 

In [59]:
lemmatizer = WordNetLemmatizer()

In [60]:
def lemmatize(word,tag):
    word = word.lower()
    wn_tag = wordnet.VERB if tag.startswith("V") else wordnet.NOUN

    return lemmatizer.lemmatize(word,wn_tag)

COUNTERS

In [61]:
C_v,C_n,C_vp,C_np = [defaultdict(int) for _ in range(4)]

INITIALIZING COUNTERS

In [62]:
for sentence in df["Sentence"]:
    tokens = word_tokenize(sentence)
    tagged = pos_tag(tokens)

    for word,tag in tagged:
        if tag.startswith("V"):
            lemma = lemmatize(word,tag)
            C_v[lemma] += 1
        elif tag.startswith("N"):
            lemma = lemmatize(word,tag)
            C_n[lemma] += 1

    for idx,(word,tag) in enumerate(tagged):

        if word.lower() == "with":

            verb_head = None
            for offset in range(1,5):
                look_left = idx - offset

                if look_left < 0:
                    break

                if tagged[look_left][1].startswith("V"):
                    verb_head = lemmatize(
                                    tagged[look_left][0],
                                    tagged[look_left][1]
                                )
                    
                    break

            noun_pobj = None
            for offset in range(1,5):
                look_right = idx + offset

                if look_right > len(tagged):
                    break

                if tagged[look_right][1].startswith("N"):
                    noun_pobj = lemmatize(
                                    tagged[look_right][0],
                                    tagged[look_right][1]
                                )
                    break
            
            if noun_pobj:
                C_np[(noun_pobj,"with")] += 1
                if verb_head:
                    C_vp[(verb_head,"with")] += 1


HINDLE ROOTH ALGORITHM 

In [63]:
def hindle_rooth(v,n,p="with"):
    if C_v[v] == 0 or C_n[n] == 0:
        return 0
    
    p_v = (C_vp.get((v,p),0)) / C_v[v]
    p_n = (C_np.get((n,p),0)) / C_n[n]
    
    if p_v == 0 or p_n == 0:
        return 0
    
    return log2((p_v*(1-p_n))/p_n)

INPUT

In [64]:
verb = "cook"
noun = "phone"  
prep = "with"

SCORE CALCULATION

In [65]:
score = hindle_rooth(verb,noun,prep)

if score > 0:
    attachment = "VERB"
else:
    attachment = "NOUN"
print(f"(v={verb}, n={noun}) → {attachment}, score={score:.3f}")


(v=cook, n=phone) → NOUN, score=-2.339
